# RQ3: Single Utterance vs EmoBERTa — Complete Pipeline (IEMOCAP)

Καθαρή, οργανωμένη έκδοση. Κάθε μοντέλο χρησιμοποιεί το σωστό dataset:
- **EmoBERTa** → context windows (`test_constructed_targetSEPonly.csv`)
- **Single Utterance** → raw utterances (`iemocap6_emoberta_test.csv`)

**Δομή:**
1. Setup (install, imports, paths)
2. Load models + datasets
3. Complexity metrics (surface + L2SCA + semantic)
4. Inference (EmoBERTa + Single, με entropy)
5. Analysis (RQ3 bins, per-metric, partial correlation)
6. Plots + Save

## 1. Setup

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn seaborn vaderSentiment pingouin

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

# IEMOCAP 6 labels — ίδια σειρά CTX & SNG (όπως στο training)
LABELS   = ['neutral','frustration','sadness','anger','excited','happiness']
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

# Column names αρχικού CSV
TEXT_COL  = 'Utterance'
LABEL_COL = 'Emotion'

BINS = ['Low','Medium-Low','Medium-High','High']
SEED = 42
np.random.seed(SEED)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ✏️ ΑΛΛΑΞΕ ΕΔΩ — paths
CHECKPOINT_CTX = '/content/drive/MyDrive/epoch_checkpoints_seed46_emoberta_roberta_iemocap/epoch_checkpoints_seed46/epoch_06'
CHECKPOINT_SNG = '/content/drive/MyDrive/fine tuned roberta iemocap_v1/robert_iemocap6_seed44_BEST'
CONTEXT_CSV    = '/content/test_constructed_targetSEPonly.csv'
SINGLE_CSV     = '/content/drive/MyDrive/iemocap6_emoberta_test.csv'
L2SCA_CSV      = '/content/drive/MyDrive/rq3_results_with_l2sca.csv'   # optional
OUTPUT_DIR     = '/content/drive/MyDrive/rq3_results_iemocap'
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_SAMPLES = None  # None = full dataset

for name, path in [('CTX ckpt',CHECKPOINT_CTX),('SNG ckpt',CHECKPOINT_SNG),
                   ('Context CSV',CONTEXT_CSV),('Single CSV',SINGLE_CSV)]:
    print(f'{name:14s}: {os.path.exists(path)}')

## 2. Load Models & Datasets

In [ ]:
# ── Load both models ──
print('Loading EmoBERTa...')
tok_ctx   = AutoTokenizer.from_pretrained(CHECKPOINT_CTX, use_fast=True, add_prefix_space=True)
model_ctx = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_CTX).eval().to(DEVICE)

print('Loading Single Utterance RoBERTa...')
tok_sng   = AutoTokenizer.from_pretrained(CHECKPOINT_SNG, use_fast=True)
model_sng = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_SNG).eval().to(DEVICE)

print('CTX label 0:', model_ctx.config.id2label.get(0), '(expected: neutral)')
print('SNG label 0:', model_sng.config.id2label.get(0), '(expected: neutral)')

In [ ]:
# ── Load + align datasets ──
# EmoBERTa: context windows
ctx_df = pd.read_csv(CONTEXT_CSV)
ctx_df['label'] = ctx_df['label'].astype(str).str.strip().str.lower()
ctx_df = ctx_df[ctx_df['label'].isin(LABELS)].copy()
ctx_df = ctx_df.sort_values(['dialogue_id','utterance_id']).reset_index(drop=True)
ctx_df['label_id'] = ctx_df['label'].map(label2id)

# Single utt: raw utterances
sng_df = pd.read_csv(SINGLE_CSV)
sng_df[LABEL_COL] = sng_df[LABEL_COL].astype(str).str.strip().str.lower()
sng_df = sng_df[sng_df[LABEL_COL].isin(LABELS)].copy()
sng_df = sng_df.sort_values(['Dialogue_ID','Utterance_ID']).reset_index(drop=True)
sng_df['label']    = sng_df[LABEL_COL]
sng_df['label_id'] = sng_df['label'].map(label2id)

print(f'Context rows: {len(ctx_df)} | Single rows: {len(sng_df)}')
align = (ctx_df['label'].values == sng_df['label'].values).mean()
print(f'Label alignment: {align:.3f}', '(OK)' if align > 0.95 else '(MISMATCH!)')

In [ ]:
# ── Tokenize context windows (EmoBERTa input) ──
# context_text_raw έχει ήδη <s> και </s> → add_special_tokens=False
print('Tokenizing context windows...')
all_ids, all_attn = [], []
texts = ctx_df['context_text_raw'].tolist()
for i in range(0, len(texts), 64):
    enc = tok_ctx(texts[i:i+64], max_length=512, truncation=True,
                  padding=False, add_special_tokens=False, return_tensors=None)
    all_ids.extend(enc['input_ids'])
    all_attn.extend(enc['attention_mask'])
ctx_df['input_ids']      = all_ids
ctx_df['attention_mask'] = all_attn
print('Done')

In [ ]:
# ── Sampling (optional) ──
if N_SAMPLES is None:
    ctx = ctx_df.copy().reset_index(drop=True)
    sng = sng_df.copy().reset_index(drop=True)
else:
    idx = []
    for lbl in LABELS:
        lbl_idx = ctx_df[ctx_df['label']==lbl].index.tolist()
        k = min(N_SAMPLES//len(LABELS), len(lbl_idx))
        idx.extend(np.random.choice(lbl_idx, k, replace=False).tolist())
    idx = sorted(idx)
    ctx = ctx_df.iloc[idx].reset_index(drop=True)
    sng = sng_df.iloc[idx].reset_index(drop=True)
print(f'Using {len(ctx)} utterances')
print(ctx['label'].value_counts())

## 3. Complexity Metrics

Όλα υπολογίζονται στο **raw target utterance** (sng) για δίκαιη σύγκριση.

In [ ]:
# ── Metric functions ──
def compute_ari(text):
    w = text.split(); nw = len(w)
    if nw == 0: return 0.0
    nc = sum(len(x) for x in w)
    ns = max(1, text.count('.')+text.count('!')+text.count('?'))
    return round(4.71*(nc/nw)+0.5*(nw/ns)-21.43, 3)

def compute_lix(text):
    w = text.split(); nw = len(w)
    if nw == 0: return 0.0
    ns = max(1, text.count('.')+text.count('!')+text.count('?'))
    lw = sum(1 for x in w if len(re.sub(r'[^a-zA-Z]','',x)) > 6)
    return round((nw/ns)+100*(lw/nw), 3)

def compute_mattr(text, window=50):
    t = text.lower().split(); n = len(t)
    if n < window: return round(len(set(t))/max(1,n), 3)
    return round(float(np.mean([len(set(t[i:i+window]))/window for i in range(n-window+1)])), 3)

SUB = {'because','although','since','while','if','when','that','which','who',
       'unless','until','after','before','though','whenever','wherever',
       'whether','whereas','even'}
COORD = {'and','but','or','nor','so','yet','for'}

def compute_dc_c_proxy(text):
    w = [x.lower() for x in re.findall(r"[a-zA-Z']+", text)]
    if not w: return 0.0
    dc = sum(1 for x in w if x in SUB)
    total = max(1, text.count('.')+text.count('!')+text.count('?')+sum(1 for x in w if x in SUB|COORD))
    return round(dc/total, 3)

STOP = {'the','a','an','is','are','was','were','be','been','being','have','has',
        'had','do','does','did','will','would','shall','should','may','might',
        'must','can','could','to','of','in','on','at','by','for','with','as',
        'from','and','but','or','so','yet','nor','this','that','it','its','i',
        'we','you','he','she','they','my','your','his','her','our','their'}

def compute_add_proxy(text):
    w = [x.lower() for x in text.split() if re.sub(r'[^a-zA-Z]','',x)]
    if len(w) < 2: return 0.0
    content = [(i,x) for i,x in enumerate(w) if x not in STOP]
    if len(content) < 2: return 0.0
    d = [content[i+1][0]-content[i][0] for i in range(len(content)-1)]
    return round(sum(d)/len(d), 3)

def normalize(s):
    mn, mx = s.min(), s.max()
    return (s-mn)/(mx-mn) if mx != mn else pd.Series([0.5]*len(s), index=s.index)

print('Metric functions defined')

In [ ]:
# ── Compute surface + proxy metrics στο raw utterance ──
text_series = sng[TEXT_COL].astype(str)

ctx['ARI']        = text_series.apply(compute_ari).values
ctx['LIX']        = text_series.apply(compute_lix).values
ctx['MATTR']      = text_series.apply(compute_mattr).values
ctx['DC_C_proxy'] = text_series.apply(compute_dc_c_proxy).values
ctx['ADD_proxy']  = text_series.apply(compute_add_proxy).values
ctx['n_words']    = text_series.apply(lambda t: len(t.split())).values

# Composite
ctx['ARI_n']     = normalize(ctx['ARI'])
ctx['MATTR_inv'] = 1 - normalize(ctx['MATTR'])
ctx['DC_n']      = normalize(ctx['DC_C_proxy'])
ctx['ADD_n']     = normalize(ctx['ADD_proxy'])
ctx['composite_full'] = (0.30*ctx['DC_n'] + 0.25*ctx['ADD_n'] +
                         0.25*ctx['ARI_n'] + 0.20*ctx['MATTR_inv']).round(4)
ctx['complexity_bin'] = pd.qcut(ctx['composite_full'], q=4, labels=BINS)

print('Surface metrics computed')
print(ctx[['ARI','LIX','MATTR','DC_C_proxy','ADD_proxy','composite_full']].describe().round(3))

In [ ]:
# ── Merge L2SCA (optional) ──
if os.path.exists(L2SCA_CSV):
    l2sca = pd.read_csv(L2SCA_CSV)[['dialogue_id','utterance_id','MLT_l2sca','MLS_l2sca','DC_C_l2sca']]
    ctx = ctx.merge(l2sca, on=['dialogue_id','utterance_id'], how='left')
    print('L2SCA merged:', ctx['MLT_l2sca'].notna().sum(), 'rows')
else:
    print('L2SCA CSV not found — skipping (proxies still available)')
    ctx['MLT_l2sca'] = ctx['MLS_l2sca'] = ctx['DC_C_l2sca'] = np.nan

In [ ]:
# ── Semantic metrics: VADER ambiguity + emotion density ──
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

def sentiment_ambiguity(text):
    return round(1 - abs(sia.polarity_scores(str(text))['compound']), 3)

EMO_WORDS = {
    'happy','happiness','joy','excited','wonderful','great','love','amazing',
    'fantastic','glad','pleased','thrilled','delighted','laugh','funny','fun',
    'sad','sorry','miss','hurt','cry','terrible','awful','unhappy','depressed',
    'regret','grief','lonely','heartbroken','disappointed',
    'angry','mad','frustrated','annoyed','irritated','furious','hate','upset',
    'ridiculous','unfair','stupid','damn','hell','wrong','unacceptable',
    'scared','afraid','worried','nervous','anxious','fear','terrified','panic',
    'concerned','stress','trouble',
    'surprised','shocked','wow','really','seriously','unbelievable','incredible',
    'unexpected','fine','okay','whatever','nothing','nobody','anyway'
}
def emotion_density(text):
    w = [x.strip('.,!?;:').lower() for x in str(text).split()]
    if not w: return 0.0
    return round(sum(1 for x in w if x in EMO_WORDS)/len(w), 3)

ctx['ambiguity']   = text_series.apply(sentiment_ambiguity).values
ctx['emo_density'] = text_series.apply(emotion_density).values
print('Semantic metrics computed')

## 4. Inference (με entropy)

In [ ]:
# ── Inference functions (αποθηκεύουν all_probs για entropy) ──
def compute_entropy(probs):
    p = np.array(probs)
    return round(-np.sum(p * np.log(p + 1e-10)), 3)

@torch.no_grad()
def infer_ctx(input_ids_list, attn_list, batch_size=16):
    preds, confs, all_probs = [], [], []
    for i in range(0, len(input_ids_list), batch_size):
        bi = input_ids_list[i:i+batch_size]
        ba = attn_list[i:i+batch_size]
        ml = max(len(x) for x in bi)
        ids  = torch.tensor([x+[1]*(ml-len(x)) for x in bi], dtype=torch.long).to(DEVICE)
        attn = torch.tensor([x+[0]*(ml-len(x)) for x in ba], dtype=torch.long).to(DEVICE)
        probs = torch.softmax(model_ctx(input_ids=ids, attention_mask=attn).logits, -1).cpu().numpy()
        for p in probs:
            pid = int(np.argmax(p))
            preds.append(id2label[pid]); confs.append(round(float(p[pid]),4)); all_probs.append(p.tolist())
    return preds, confs, all_probs

@torch.no_grad()
def infer_sng(texts, batch_size=16):
    preds, confs, all_probs = [], [], []
    for i in range(0, len(texts), batch_size):
        enc = tok_sng(texts[i:i+batch_size], max_length=256, truncation=True,
                      padding=True, return_tensors='pt').to(DEVICE)
        probs = torch.softmax(model_sng(**enc).logits, -1).cpu().numpy()
        for p in probs:
            pid = int(np.argmax(p))
            preds.append(id2label[pid]); confs.append(round(float(p[pid]),4)); all_probs.append(p.tolist())
    return preds, confs, all_probs

print('Inference functions defined')

In [ ]:
# ── Run EmoBERTa inference ──
print('EmoBERTa inference...')
preds, confs, probs = infer_ctx(ctx['input_ids'].tolist(), ctx['attention_mask'].tolist())
ctx['pred_ctx']    = preds
ctx['conf_ctx']    = confs
ctx['entropy_ctx'] = [compute_entropy(p) for p in probs]
ctx['correct_ctx'] = (ctx['pred_ctx'] == ctx['label']).astype(int)
ctx['pred_ctx_id'] = ctx['pred_ctx'].map(label2id)

print(f'EmoBERTa  Acc={ctx["correct_ctx"].mean():.3f}  '
      f'W-F1={f1_score(ctx["label_id"], ctx["pred_ctx_id"], average="weighted"):.3f}')

In [ ]:
# ── Run Single Utterance inference ──
print('Single utterance inference...')
preds, confs, probs = infer_sng(sng[TEXT_COL].astype(str).tolist())
ctx['pred_sng']    = preds
ctx['conf_sng']    = confs
ctx['entropy_sng'] = [compute_entropy(p) for p in probs]
ctx['correct_sng'] = (ctx['pred_sng'] == ctx['label']).astype(int)
ctx['pred_sng_id'] = ctx['pred_sng'].map(label2id)

print(f'Single    Acc={ctx["correct_sng"].mean():.3f}  '
      f'W-F1={f1_score(ctx["label_id"], ctx["pred_sng_id"], average="weighted"):.3f}')

# Derived metrics
ctx['gain_per_utt']      = ctx['correct_ctx'] - ctx['correct_sng']
ctx['entropy_reduction'] = ctx['entropy_sng'] - ctx['entropy_ctx']
df = ctx  # alias για συντομία
print(f'\nMean gain: +{df["gain_per_utt"].mean():.3f}')

## 5. Analysis

In [ ]:
# ── RQ3: F1 per complexity bin ──
print('='*58)
print('RQ3: F1 PER COMPLEXITY BIN (composite_full)')
print('='*58)
print(f'{"Bin":15s}  {"Single":8s}  {"EmoBERTa":8s}  {"Gain":7s}  n')
print('-'*58)
rq3_rows = []
for b in BINS:
    sub = df[df['complexity_bin']==b]
    if len(sub)==0: continue
    fc = f1_score(sub['label_id'], sub['pred_ctx_id'], average='weighted', zero_division=0)
    fs = f1_score(sub['label_id'], sub['pred_sng_id'], average='weighted', zero_division=0)
    print(f'{b:15s}  {fs:.3f}     {fc:.3f}     {fc-fs:+.3f}   {len(sub)}')
    rq3_rows.append({'bin':b,'f1_single':round(fs,3),'f1_emoberta':round(fc,3),'gain':round(fc-fs,3),'n':len(sub)})
rq3_df = pd.DataFrame(rq3_rows)

df['bin_rank'] = df['complexity_bin'].map({b:i+1 for i,b in enumerate(BINS)})
r, p = stats.pearsonr(df['bin_rank'], df['gain_per_utt'])
sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
print(f'\nCorrelation bin_rank vs gain: r={r:+.3f} p={p:.4f} {sig}')

In [ ]:
# ── Full correlation table: όλες οι metrics vs gain ──
metrics = {
    'LIX':'LIX', 'ARI':'ARI', 'MATTR':'MATTR', 'ADD_proxy':'ADD_proxy',
    'DC_C_proxy':'DC_C_proxy', 'composite':'composite_full',
    'MLT_l2sca':'MLT_l2sca', 'MLS_l2sca':'MLS_l2sca', 'DC_C_l2sca':'DC_C_l2sca',
    'entropy_reduction':'entropy_reduction', 'ambiguity':'ambiguity', 'emo_density':'emo_density',
}
print('='*58)
print('FULL CORRELATION TABLE — metrics vs context gain')
print('='*58)
corr_rows = []
for name, col in metrics.items():
    if col not in df.columns or df[col].isna().all():
        print(f'  {name:20s}: MISSING'); continue
    valid = df[col].notna() & df['gain_per_utt'].notna()
    if valid.sum() < 10:
        print(f'  {name:20s}: too few'); continue
    r, p = stats.pearsonr(df.loc[valid,col], df.loc[valid,'gain_per_utt'])
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
    mark = '  <- strong' if abs(r)>0.12 and p<0.001 else ''
    print(f'  {name:20s}  r={r:+.3f}  p={p:.4f}  {sig}{mark}')
    corr_rows.append({'metric':name,'r':round(r,3),'p':round(p,4),'sig':sig})
corr_df = pd.DataFrame(corr_rows)

In [ ]:
# ── Per-metric bin analysis ──
def bin_analysis(metric_col, label):
    try:
        tmp_bin = pd.qcut(df[metric_col], q=4, labels=BINS, duplicates='drop')
    except Exception as e:
        print(f'{label}: cannot bin'); return
    print(f'\n{label}:')
    for b in BINS:
        sub = df[tmp_bin==b]
        if len(sub) < 10: continue
        fc = f1_score(sub['label_id'], sub['pred_ctx_id'], average='weighted', zero_division=0)
        fs = f1_score(sub['label_id'], sub['pred_sng_id'], average='weighted', zero_division=0)
        print(f'  {b:13s}  single={fs:.3f}  emoberta={fc:.3f}  gain={fc-fs:+.3f}')
    rank = tmp_bin.map({b:i+1 for i,b in enumerate(BINS)})
    valid = rank.notna()
    r, p = stats.pearsonr(rank[valid], df.loc[valid,'gain_per_utt'])
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
    print(f'  r={r:+.3f} p={p:.4f} {sig}')

print('PER-METRIC BIN ANALYSIS')
print('='*58)
for col, lbl in [('LIX','LIX'),('ARI','ARI'),('ADD_proxy','ADD proxy'),
                 ('MLT_l2sca','MLT (L2SCA)'),('MLS_l2sca','MLS (L2SCA)')]:
    if col in df.columns and df[col].notna().sum() > 10:
        bin_analysis(col, lbl)

In [ ]:
# ── Entropy analysis ──
print('='*58)
print('ENTROPY ANALYSIS')
print('='*58)
print(f'Mean entropy EmoBERTa:   {df["entropy_ctx"].mean():.3f}')
print(f'Mean entropy Single-utt: {df["entropy_sng"].mean():.3f}')
print()
print(f'{"Bin":15s}  {"EmoBERTa":10s}  {"Single":10s}  {"Reduction":10s}')
for b in BINS:
    sub = df[df['complexity_bin']==b]
    if len(sub)==0: continue
    print(f'{b:15s}  {sub["entropy_ctx"].mean():.3f}       {sub["entropy_sng"].mean():.3f}       {sub["entropy_reduction"].mean():.3f}')

r, p = stats.pearsonr(df['entropy_reduction'], df['gain_per_utt'])
sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
print(f'\nentropy_reduction vs gain: r={r:+.3f} p={p:.4f} {sig}')

In [ ]:
# ── Partial correlation (length disentanglement) ──
import pingouin as pg
print('='*58)
print('PARTIAL CORRELATION — controlling for n_words')
print('='*58)
for metric in ['LIX','composite_full','MLT_l2sca','MLS_l2sca']:
    if metric not in df.columns or df[metric].isna().all(): continue
    rs, ps = stats.pearsonr(df[metric].fillna(0), df['gain_per_utt'])
    res = pg.partial_corr(data=df, x=metric, y='gain_per_utt', covar='n_words')
    rp = res['r'].values[0]
    pcol = 'p-val' if 'p-val' in res.columns else 'pval'
    pp = res[pcol].values[0]
    sg = '***' if ps<0.001 else '**' if ps<0.01 else '*' if ps<0.05 else 'n.s.'
    gp = '***' if pp<0.001 else '**' if pp<0.01 else '*' if pp<0.05 else 'n.s.'
    print(f'{metric:18s}  simple r={rs:+.3f} {sg:4s}  partial r={rp:+.3f} {gp}')

## 6. Plots & Save

In [ ]:
# ── Main RQ3 plots ──
COLORS = ['#1D9E75','#378ADD','#BA7517','#D85A30']
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Grouped bar
ax = axes[0]
x = np.arange(len(BINS)); w = 0.35
ax.bar(x-w/2, rq3_df['f1_single'], w, label='Single Utterance', color='#378ADD', alpha=0.85)
ax.bar(x+w/2, rq3_df['f1_emoberta'], w, label='EmoBERTa', color='#1D9E75', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(BINS); ax.set_ylim(0,1)
ax.set_ylabel('Weighted F1'); ax.set_title('RQ3: Single vs EmoBERTa per Complexity Bin')
ax.legend()

# Gain line
ax = axes[1]
ax.plot(BINS, rq3_df['f1_single'], 'o-', color='#378ADD', lw=2, ms=8, label='Single')
ax.plot(BINS, rq3_df['f1_emoberta'], 's-', color='#1D9E75', lw=2, ms=8, label='EmoBERTa')
ax.fill_between(BINS, rq3_df['f1_single'], rq3_df['f1_emoberta'], alpha=0.15, color='#1D9E75')
ax.set_ylim(0,1); ax.set_ylabel('Weighted F1'); ax.set_title('Context Gain (shaded)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/rq3_main.png', dpi=150); plt.show()

In [ ]:
# ── Correlation bar chart ──
valid = corr_df.dropna()
colors = ['#085041' if r<-0.10 else '#378ADD' if r<0 else '#D85A30' for r in valid['r']]
fig, ax = plt.subplots(figsize=(10,5))
ax.barh(valid['metric'], valid['r'], color=colors, edgecolor='white')
ax.axvline(0, color='gray', lw=0.8, ls='--')
for i, (m, r) in enumerate(zip(valid['metric'], valid['r'])):
    ax.text(r+(0.003 if r>=0 else -0.003), i, f'{r:+.3f}', va='center',
            ha='left' if r>=0 else 'right', fontsize=9)
ax.set_xlabel('Pearson r (vs context gain)')
ax.set_title('All Metrics vs Context Gain (IEMOCAP)')
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/correlation_all.png', dpi=150); plt.show()

In [ ]:
# ── Save ──
save_cols = ['dialogue_id','utterance_id','label','label_id',
             'ARI','LIX','MATTR','DC_C_proxy','ADD_proxy','composite_full','n_words',
             'MLT_l2sca','MLS_l2sca','DC_C_l2sca',
             'ambiguity','emo_density','complexity_bin',
             'pred_ctx','conf_ctx','entropy_ctx','correct_ctx',
             'pred_sng','conf_sng','entropy_sng','correct_sng',
             'gain_per_utt','entropy_reduction']
save_cols = [c for c in save_cols if c in df.columns]
df[save_cols].to_csv(f'{OUTPUT_DIR}/rq3_full_results.csv', index=False)
rq3_df.to_csv(f'{OUTPUT_DIR}/rq3_summary.csv', index=False)
corr_df.to_csv(f'{OUTPUT_DIR}/correlation_full.csv', index=False)

print('Saved to', OUTPUT_DIR)
print(f'\nEmoBERTa W-F1: {f1_score(df["label_id"],df["pred_ctx_id"],average="weighted"):.3f}')
print(f'Single   W-F1: {f1_score(df["label_id"],df["pred_sng_id"],average="weighted"):.3f}')
display(rq3_df)
display(corr_df)